In [ ]:
import pickle

import healpy as hp
import matplotlib.pyplot as plt
from pyoperators import MPI

from qubic.lib.QskySim import get_angular_profile, get_noise_invcov_profile

comm = MPI.COMM_WORLD

In [ ]:
new = pickle.load(open("test_w_planck_noise/Dict/test_None.pkl", "rb"))
old = pickle.load(open("test_wo_planck_noise/Dict/test_None.pkl", "rb"))

print(new["parameters"]["PLANCK"]["external_data"])
print(old["parameters"]["PLANCK"]["external_data"])

label_new = "QUBIC + Planck w noise"
label_old = "QUBIC + Planck wo noise"

In [ ]:
new["parameters"]

In [ ]:
convergence_new = new["convergence"]
convergence_old = old["convergence"]

plt.plot(convergence_new, label=label_new)
plt.plot(convergence_old, label=label_old)
plt.xlabel('iterations')
plt.ylabel('normalized residual')
plt.title('Convergence of the PCG algorithm')
plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
maps_in_new = new['maps_in']
maps_rec_new = new['maps']
residuals_new = new['maps_noise']

maps_in_old = old['maps_in']
maps_rec_old = old['maps']
residuals_old = old['maps_noise']

In [ ]:
stk = ['I', 'Q', 'U']
inu = 1
istk = 1
n_sig = 3
reso = 30
min = -8
max = 8

plt.figure(figsize=(10, 12))
print(label_new)
k=1
hp.gnomview(maps_in_new[inu, :, istk], cmap='jet', rot=new['center'], min=min, max=max, reso=reso, sub=(1, 3, k), title=f'{stk[istk]} - Input', notext=True)
hp.gnomview(maps_rec_new[inu, :, istk], cmap='jet', rot=new['center'], min=min, max=max, reso=reso, sub=(1, 3, k+1), title=f'{stk[istk]} - Output', notext=True)
hp.gnomview(residuals_new[inu, :, istk], cmap='jet', rot=new['center'], min=min/2, max=max/2, reso=reso, sub=(1, 3, k+2), title=f'{stk[istk]} - Residual', notext=True)
k+=3

plt.figure()
plt.plot(hp.anafast(maps_in_new[inu, :, 0]), label="Input")
plt.plot(hp.anafast(maps_rec_new[inu, :, 0]), label="Reconstructed")
plt.legend()
plt.xlabel("l")
plt.ylabel("Cl")
plt.title(label_new)

plt.figure()
plt.plot(hp.anafast(maps_in_new[inu, :, 0] - maps_rec_new[inu, :, 0]), label="Residual")
plt.legend()
plt.xlabel("l")
plt.ylabel("Cl")
plt.title(label_new)

In [ ]:
plt.figure(figsize=(10, 12))
print(label_old)
k=1
hp.gnomview(maps_in_old[inu, :, istk], cmap='jet', rot=old['center'], min=min, max=max, reso=reso, sub=(1, 3, k), title=f'{stk[istk]} - Input', notext=True)
hp.gnomview(maps_rec_old[inu, :, istk], cmap='jet', rot=old['center'], min=min, max=max, reso=reso, sub=(1, 3, k+1), title=f'{stk[istk]} - Output', notext=True)
hp.gnomview(residuals_old[inu, :, istk], cmap='jet', rot=old['center'], min=min/2, max=max/2, reso=reso, sub=(1, 3, k+2), title=f'{stk[istk]} - Residual', notext=True)
k+=3

plt.figure()
plt.plot(hp.anafast(maps_in_old[inu, :, 0]), label="Input")
plt.plot(hp.anafast(maps_rec_old[inu, :, 0]), label="Reconstructed")
plt.legend()
plt.xlabel("l")
plt.ylabel("Cl")
plt.title(label_old)

plt.figure()
plt.plot(hp.anafast(maps_in_old[inu, :, 0] - maps_rec_old[inu, :, 0]), label="Residual")
plt.legend()
plt.xlabel("l")
plt.ylabel("Cl")
plt.title(label_old)

In [ ]:
get_angular_profile(residuals_new[0], doplot=True, allstokes=True, nbins=80, thmax=20, label=label_new)
get_angular_profile(residuals_old[0], doplot=True, allstokes=True, nbins=80, thmax=20, label=label_old)
plt.title("Residuals")
plt.show()

In [ ]:
get_noise_invcov_profile(residuals_new[0], new["coverage"], doplot=True, allstokes=True, label=label_new)
plt.figure()
get_noise_invcov_profile(residuals_old[0], old["coverage"], doplot=True, allstokes=True, label=label_old)
plt.show()

# Test RMS Planck

In [ ]:
from qubic.lib.MapMaking.FrequencyMapMaking.Qfmm import PipelineFrequencyMapMaking

pipeline = PipelineFrequencyMapMaking(comm, "test/Dict/test_None.pkl", new["parameters"])

In [ ]:
planck_maps = pipeline.externaldata.maps
planck_noise = pipeline.externaldata.maps_noise

print(planck_maps.shape)

In [ ]:
plt.figure(figsize=(10, 12))
inu = -3
k=1
hp.gnomview(planck_maps[inu, :, istk], cmap='jet', rot=new['center'], reso=reso, sub=(1, 3, k), title=f'{stk[istk]} - Input', notext=True)
hp.gnomview(planck_maps[inu, :, istk], cmap='jet', rot=new['center'], reso=reso, sub=(1, 3, k+1), title=f'{stk[istk]} - Output', notext=True)
hp.gnomview(planck_maps[inu, :, istk], cmap='jet', rot=new['center'], reso=reso, sub=(1, 3, k+2), title=f'{stk[istk]} - Residual', notext=True)

In [ ]:
get_angular_profile(planck_noise[inu], doplot=True, allstokes=True, nbins=80, thmax=40, label="Planck only")
get_angular_profile(residuals_new[0], doplot=True, allstokes=True, nbins=80, thmax=40, label=label_new)
get_angular_profile(residuals_old[0], doplot=True, allstokes=True, nbins=80, thmax=40, label=label_old)
plt.title("Residuals")
plt.show()